# 01. Data Preparation

В этом ноутбуке загружаем исходные CSV-файлы, выбираем один дневной ряд продаж, добавляем внешние факторы, обрабатываем пропуски и выбросы, а затем сохраняем подготовленный датасет для следующих этапов.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    add_calendar_features,
    handle_outliers_iqr,
    load_raw_data,
    prepare_daily_series,
)
from src.visualization import save_plot

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
IMAGES = PROJECT_ROOT / "images"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
IMAGES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)

## 2. Load raw data

Перед запуском убедитесь, что CSV-файлы из Kaggle лежат в папке `data/raw/`. Автоматическое скачивание датасета в проекте не используется.

In [ ]:
data = load_raw_data(DATA_RAW)

for name, df_raw in data.items():
    print(f"{name}: {df_raw.shape}")

train = data["train"]
stores = data.get("stores")
oil = data.get("oil")
holidays = data.get("holidays_events")
transactions = data.get("transactions")

## 3. Select time series

Основной ряд проекта: дневные продажи магазина `store_nbr = 1` и товарной группы `GROCERY I`. Если такой группы нет, выбирается самая продаваемая товарная группа для магазина 1.

In [ ]:
daily_sales = prepare_daily_series(
    train=train,
    stores=stores,
    oil=oil,
    holidays=holidays,
    transactions=transactions,
    store_nbr=1,
    family="GROCERY I",
)

selected_family = daily_sales.attrs.get("selected_family", "unknown")
selected_store = daily_sales.attrs.get("selected_store_nbr", 1)

print(f"Selected store: {selected_store}")
print(f"Selected family: {selected_family}")
daily_sales.head()

## 4. Merge external factors

Внешние факторы добавляются на этапе подготовки ряда: промо, цена нефти, транзакции и праздничные дни, если соответствующие файлы доступны.

In [ ]:
print("Columns:", daily_sales.columns.tolist())
print("Shape:", daily_sales.shape)
display(daily_sales.head())
display(daily_sales.tail())

## 5. Handle missing dates and outliers

Пропущенные даты уже восстановлены внутри `prepare_daily_series`. Выбросы обрабатываются методом IQR winsorization: экстремальные значения ограничиваются, но строки не удаляются.

In [ ]:
daily_sales = handle_outliers_iqr(daily_sales, target_col="sales")

adjusted_points = int((daily_sales["sales"] != daily_sales["sales_clean"]).sum())
print(f"Adjusted outlier points: {adjusted_points}")
display(daily_sales[["sales", "sales_clean"]].describe())

## 6. Add calendar features

In [ ]:
daily_sales = add_calendar_features(daily_sales)
display(daily_sales.head())

## 7. Save processed dataset

In [ ]:
processed_path = DATA_PROCESSED / "daily_sales_series.csv"
daily_sales.to_csv(processed_path, index=False)

print(f"Saved to: {processed_path}")
print("Shape:", daily_sales.shape)
display(daily_sales.head())
display(daily_sales.tail())
display(daily_sales.describe(include="all"))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily_sales["date"], daily_sales["sales"], label="sales")
ax.plot(daily_sales["date"], daily_sales["sales_clean"], label="sales_clean")
ax.set_title("Daily Sales: Raw vs Clean")
ax.set_xlabel("date")
ax.set_ylabel("sales")
ax.legend()
ax.grid(True)
save_plot(IMAGES / "prepared_sales_clean.png")
plt.show()

## 8. Key findings

- Подготовлен единый дневной ряд продаж для одного магазина и одной товарной группы.
- Пропущенные даты восстановлены, продажи по отсутствующим датам заполнены нулем.
- В ряд добавлены доступные внешние факторы: промо, праздники, транзакции и цена нефти.
- Создана очищенная целевая переменная `sales_clean`, которая будет использоваться в моделях.
- Подготовленный датасет сохранен в `data/processed/daily_sales_series.csv`.